[![cloudevel](../img/cloudevel.png)](https://cloudevel.com)

# Solución: despliegue en Cloud Run

**Cómo leer este cuaderno.** Cada ejercicio reproduce el enunciado del cuaderno original y presenta su solución.

* Las celdas **sin etiqueta** se ejecutan en cualquier equipo, sin credenciales ni cuenta de *GitHub*. Escriben solo en carpetas temporales.
* Las celdas con la etiqueta `skip-execution` necesitan una cuenta de *GitHub*, un proyecto de nube o un servidor; la integración continua (*CI*) no las ejecuta. Debajo de cada una se describe la salida esperada.
* Ningún valor real aparece en el código: los datos de cada persona se leen de variables de entorno o se sustituyen por los marcadores indicados.

**Estado de alineación.** Esta solución se escribió contra el contenido de `main` (`dff2150`). La rama de la integración continua `ci/cuadernos-16-17-2711` (`fe1cc48`) solo cambió metadatos: las celdas de código del cuaderno 16 con `preparar_proyecto.sh`, `configurar_wif.sh`, `desplegar_manual.sh` y `limpiar.sh` llevan la etiqueta `skip-execution`. **Pendiente de alineación** si esa rama cambia después celdas o *workflows*; las comprobaciones de este cuaderno leen los *workflows* reales y fallarán de forma visible si cambian los nombres de pasos o de variables.

**Requisitos para las celdas con etiqueta.** Un proyecto de *Google Cloud* con facturación habilitada, propio de cada persona; el *SDK* `gcloud` con la sesión iniciada como propietaria del proyecto; `gh` con la sesión iniciada; y una imagen publicada en *GHCR* por el *CI* de `main` con el paquete en modo público. Los identificadores del proyecto y de la región se leen de las variables de entorno `GCP_PROJECT_ID` y `GCP_REGION`; ningún valor real se escribe aquí.

La siguiente celda importará el módulo `utilidades` (que está en esta misma carpeta) y localizará la raíz del repositorio. Todas las comprobaciones locales de este cuaderno leen archivos desde esa raíz.

In [ ]:
import utilidades as u

RAIZ = u.raiz_repositorio()
RAIZ.name

## Ejercicio 1. Preparar el proyecto, configurar WIF y cargar las variables del entorno cloudrun

Definir en la sesión `GCP_PROJECT_ID` y `GCP_REGION`, ejecutar `preparar_proyecto.sh` y `configurar_wif.sh`, y cargar en el entorno `cloudrun` las variables `GCP_PROJECT_ID`, `GCP_REGION`, `AR_REPOSITORY`, `CLOUD_RUN_SERVICE`, `GCP_WIF_PROVIDER`, `GCP_DEPLOY_SA` y `GCP_RUNTIME_SA`.

El enunciado no dice qué valor lleva cada variable. Los guiones los derivan de `GCP_PROJECT_ID` y de unos nombres por defecto (`variables.sh`), y `configurar_wif.sh` imprime el valor de `GCP_WIF_PROVIDER`. La siguiente celda ejecuta `variables.sh` con valores **de ejemplo** (que no corresponden a ningún proyecto real) para ver cómo se derivan las demás.

La siguiente celda ejecutará `api/deploy/gcp/variables.sh` con `bash` y valores de ejemplo, e imprimirá los nombres derivados. Solo lee variables de entorno: no invoca `gcloud` porque `GCP_PROJECT_ID` está definida.

In [ ]:
import subprocess

EJEMPLO = {"GCP_PROJECT_ID": "proyecto-ejemplo-123", "GCP_REGION": "ejemplo-norte1"}
guion = (
    f'source "{RAIZ}/api/deploy/gcp/variables.sh"; '
    'for n in GCP_PROJECT_ID GCP_REGION AR_REPOSITORY CLOUD_RUN_SERVICE GCP_DEPLOY_SA GCP_RUNTIME_SA IMAGEN_BASE; '
    'do printf "%s=%s\\n" "$n" "${!n}"; done'
)
salida = subprocess.run(["bash", "-c", guion], env={"PATH": "/usr/bin:/bin", **EJEMPLO},
                        capture_output=True, text=True, check=True).stdout
DERIVADAS = dict(linea.split("=", 1) for linea in salida.splitlines())
print(salida)
assert DERIVADAS["AR_REPOSITORY"] == "cd441" and DERIVADAS["CLOUD_RUN_SERVICE"] == "cd441-api"
assert DERIVADAS["GCP_DEPLOY_SA"] == "cd441-deployer@proyecto-ejemplo-123.iam.gserviceaccount.com"
assert DERIVADAS["GCP_RUNTIME_SA"] == "cd441-runtime@proyecto-ejemplo-123.iam.gserviceaccount.com"
assert DERIVADAS["IMAGEN_BASE"] == "ejemplo-norte1-docker.pkg.dev/proyecto-ejemplo-123/cd441/cd441-api"

La tabla de valores es la siguiente:

| Variable | Valor | Origen |
|---|---|---|
| `GCP_PROJECT_ID` | identificador del proyecto de cada persona | se define en la sesión |
| `GCP_REGION` | región elegida por quien cursa | se define en la sesión |
| `AR_REPOSITORY` | `cd441` | valor por defecto de `variables.sh` |
| `CLOUD_RUN_SERVICE` | `cd441-api` | valor por defecto de `variables.sh` |
| `GCP_DEPLOY_SA` | `cd441-deployer@<proyecto>.iam.gserviceaccount.com` | derivado |
| `GCP_RUNTIME_SA` | `cd441-runtime@<proyecto>.iam.gserviceaccount.com` | derivado |
| `GCP_WIF_PROVIDER` | `projects/<número>/locations/global/workloadIdentityPools/github/providers/cd441-actions` | lo imprime `configurar_wif.sh` |

La siguiente celda ejecutará el paso real `Comprobar y validar variables` de `deploy-cloud-run.yml` con cuatro configuraciones del entorno y comprobará su comportamiento: sin ninguna variable se omite el despliegue con éxito; con algunas variables falla; con todas y formato válido se configura; y con un identificador de proyecto con caracteres peligrosos falla. Los valores de `GCP_WIF_PROVIDER` y las cuentas son marcadores de ejemplo.

In [ ]:
WORKFLOW = RAIZ / ".github" / "workflows" / "deploy-cloud-run.yml"
PASO = "Comprobar y validar variables"
completo = {**EJEMPLO, "AR_REPOSITORY": "cd441", "CLOUD_RUN_SERVICE": "cd441-api",
            "GCP_WIF_PROVIDER": "marcador", "GCP_DEPLOY_SA": "marcador", "GCP_RUNTIME_SA": "marcador"}

vacio = u.ejecutar_paso_run(WORKFLOW, "desplegar", PASO, {})
parcial = u.ejecutar_paso_run(WORKFLOW, "desplegar", PASO, {"GCP_REGION": "ejemplo-norte1"})
valido = u.ejecutar_paso_run(WORKFLOW, "desplegar", PASO, completo)
peligroso = u.ejecutar_paso_run(WORKFLOW, "desplegar", PASO, {**completo, "GCP_PROJECT_ID": "x; rm -rf /"})

for nombre, resultado in (("sin variables", vacio), ("parcial", parcial), ("completo", valido), ("peligroso", peligroso)):
    print(f"{nombre:14} código {resultado['codigo']} salidas {resultado['salidas']}")

assert (vacio["codigo"], vacio["salidas"]) == (0, {"configurado": "false"})
assert parcial["codigo"] == 1 and "GCP_PROJECT_ID" in parcial["texto"]
assert (valido["codigo"], valido["salidas"]) == (0, {"configurado": "true"})
assert peligroso["codigo"] == 1 and "no válido" in peligroso["texto"]

Este comportamiento es el que permite que el repositorio del curso (sin proyecto de *GCP*) se etiquete sin que el despliegue falle: el *workflow* termina en éxito y solo avisa en su resumen.

La siguiente celda preparará el proyecto (*API* habilitadas, repositorio de *Artifact Registry* y las dos cuentas de servicio sin llaves) y configurará *Workload Identity Federation* para este repositorio y el entorno `cloudrun`. Imprimirá el valor de `GCP_WIF_PROVIDER`.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
: "${GCP_PROJECT_ID:?Defina GCP_PROJECT_ID}" "${GCP_REGION:?Defina GCP_REGION}"
REPO="$(gh repo view --json nameWithOwner --jq .nameWithOwner)"
bash api/deploy/gcp/preparar_proyecto.sh
GITHUB_REPO="${REPO}" bash api/deploy/gcp/configurar_wif.sh

**Salida esperada.** Los guiones son idempotentes: si un recurso ya existe, no lo vuelven a crear. Al final se lee `Variable GCP_WIF_PROVIDER del entorno «cloudrun»:` seguida de una línea `projects/<número del proyecto>/locations/global/workloadIdentityPools/github/providers/cd441-actions`. No se genera ninguna llave *JSON*.

La siguiente celda cargará las siete variables en el entorno `cloudrun`. `GCP_WIF_PROVIDER` se toma de la variable de entorno del mismo nombre (se copia de la salida anterior); las demás se derivan con las mismas reglas de `variables.sh`.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
: "${GCP_PROJECT_ID:?}" "${GCP_REGION:?}" "${GCP_WIF_PROVIDER:?Copie el valor que imprimió configurar_wif.sh}"
AR_REPOSITORY="${AR_REPOSITORY:-cd441}"
CLOUD_RUN_SERVICE="${CLOUD_RUN_SERVICE:-cd441-api}"
GCP_DEPLOY_SA="cd441-deployer@${GCP_PROJECT_ID}.iam.gserviceaccount.com"
GCP_RUNTIME_SA="cd441-runtime@${GCP_PROJECT_ID}.iam.gserviceaccount.com"
for NOMBRE in GCP_PROJECT_ID GCP_REGION AR_REPOSITORY CLOUD_RUN_SERVICE GCP_WIF_PROVIDER GCP_DEPLOY_SA GCP_RUNTIME_SA; do
  gh variable set "${NOMBRE}" --env cloudrun --body "${!NOMBRE}"
done
gh variable list --env cloudrun

**Salida esperada.** Siete variables en el entorno `cloudrun`. Ninguna es un secreto: la autenticación usa *OIDC* y no hay llave que guardar.

## Ejercicio 2. Primer despliegue manual y comprobación de GET /health

Ejecutar el primer despliegue manual y abrir la *URL* del servicio, comprobando `GET /health`.

La siguiente celda ejecutará `desplegar_manual.sh`, que construye la imagen del *commit* actual, la sube a *Artifact Registry*, despliega el servicio con acceso público (`--allow-unauthenticated`, que solo puede conceder la persona propietaria) y llama a `comprobar.sh`.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
bash api/deploy/gcp/desplegar_manual.sh

**Salida esperada.** Al final aparece `Servicio: cd441-api · URL: https://cd441-api-<hash>.<dominio de Cloud Run>`, la lista de revisiones recientes, la imagen en *Artifact Registry* con la etiqueta igual al *SHA* del *commit* y la respuesta `{"almacen":"memoria","status":"ok","version":"<SHA del commit>"}`.

La siguiente celda obtendrá la *URL* del servicio y comprobará la respuesta de `/health` con `curl`. Fallará si la respuesta no es *HTTP* 200.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
URL="$(gcloud run services describe "${CLOUD_RUN_SERVICE:-cd441-api}" \
  --project "${GCP_PROJECT_ID}" --region "${GCP_REGION}" --format 'value(status.url)')"
curl -fsS "${URL}/health"
echo

**Salida esperada.** Un *JSON* con `"status":"ok"`, `"almacen":"memoria"` (no hay base de datos en *Cloud Run*) y la versión. Con la política de organización «Domain restricted sharing», el acceso público puede ser rechazado; en ese caso se usa un proyecto fuera de esa política.

## Ejercicio 3. Desplegar con el workflow y comprobar la imagen y la versión

Disparar el *workflow* `Despliegue en Cloud Run` (después de un *CI* verde en `main`) y verificar que la revisión usa la imagen promocionada desde *GHCR* con el *SHA* del *commit*, y que `APP_VERSION` en `/health` coincide con ese *SHA*.

El enunciado no explica cómo verificar «la revisión usa la imagen». La respuesta es leer la imagen del contenedor de la revisión con `gcloud run services describe` y compararla con el nombre esperado: `<región>-docker.pkg.dev/<proyecto>/<repositorio>/<servicio>:<SHA>`.

La siguiente celda calculará, con las mismas reglas del *workflow*, el nombre de la imagen de *GHCR* (en minúsculas) y el de *Artifact Registry* para un *SHA* y un repositorio de ejemplo. Después comprobará con `assert` que el nombre de la imagen en *Artifact Registry* conserva la etiqueta del *SHA* y que `/health` debe devolver ese mismo valor.

In [ ]:
SHA = "0123456789abcdef0123456789abcdef01234567"   # ejemplo
REPOSITORIO = "Propietario/cd441"                   # ejemplo (con mayúsculas)

imagen_ghcr = f"ghcr.io/{REPOSITORIO.lower()}/api:{SHA}"
registro = f"{EJEMPLO['GCP_REGION']}-docker.pkg.dev/{EJEMPLO['GCP_PROJECT_ID']}/cd441"
imagen_ar = f"{registro}/cd441-api:{SHA}"
print(imagen_ghcr)
print(imagen_ar)
assert imagen_ghcr.endswith(f"/api:{SHA}") and imagen_ghcr == imagen_ghcr.lower()
assert imagen_ar.split(":")[-1] == imagen_ghcr.split(":")[-1] == SHA

texto = WORKFLOW.read_text(encoding="utf-8")
assert 'IMAGEN_GHCR=ghcr.io/${REPOSITORIO,,}/api:${GITHUB_SHA}' in texto
assert 'IMAGEN_AR=${REGISTRO}/${CLOUD_RUN_SERVICE}:${GITHUB_SHA}' in texto
assert 'APP_VERSION=${GITHUB_SHA}' in texto

La siguiente celda disparará el *workflow* de despliegue, esperará su resultado y comparará tres valores: el *SHA* de `main`, la imagen de la revisión y la versión que informa `/health`. Requiere que el *CI* de `main` haya publicado la imagen en *GHCR* y que el paquete sea público.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
SHA="$(git rev-parse HEAD)"
gh workflow run "Despliegue en Cloud Run" --ref main
sleep 10
ID="$(gh run list --workflow "Despliegue en Cloud Run" --limit 1 --json databaseId --jq '.[0].databaseId')"
gh run watch "${ID}" --exit-status
SERVICIO="${CLOUD_RUN_SERVICE:-cd441-api}"
IMAGEN="$(gcloud run services describe "${SERVICIO}" --project "${GCP_PROJECT_ID}" --region "${GCP_REGION}" \
  --format 'value(spec.template.spec.containers[0].image)')"
URL="$(gcloud run services describe "${SERVICIO}" --project "${GCP_PROJECT_ID}" --region "${GCP_REGION}" \
  --format 'value(status.url)')"
VERSION="$(curl -fsS "${URL}/health" | python3 -c 'import json,sys; print(json.load(sys.stdin)["version"])')"
echo "SHA de main : ${SHA}"
echo "Imagen      : ${IMAGEN}"
echo "Versión     : ${VERSION}"
[ "${IMAGEN##*:}" = "${SHA}" ] && [ "${VERSION}" = "${SHA}" ] && echo "COINCIDEN"

**Salida esperada.** Las tres líneas muestran el mismo *SHA* de 40 caracteres (la imagen lo lleva tras los dos puntos) y la última línea dice `COINCIDEN`. Si el paso de promoción falla con «No existe ghcr.io/…», hay que esperar al *CI* de `main` y hacer público el paquete.

## Ejercicio 4. Reconocer en el workflow el paso de WIF, la promoción y la ausencia de acceso público

Señalar, en el *workflow*, el paso de *WIF*, el de promoción (sin reconstruir) y la ausencia de `--allow-unauthenticated`.

La siguiente celda localizará esos tres elementos en `deploy-cloud-run.yml` y los comprobará con `assert`: el paso con `google-github-actions/auth` y sus parámetros `workload_identity_provider` y `service_account` (sin `credentials_json`); el paso de promoción con `docker pull`, `docker tag` y `docker push` y sin `docker build`; y la ausencia de `--allow-unauthenticated` en cualquier línea que no sea un comentario.

In [ ]:
wf = u.cargar_workflow(WORKFLOW)
pasos_cr = u.pasos(wf, "desplegar")

auth = next(p for p in pasos_cr if p.get("uses", "").startswith("google-github-actions/auth@"))
print("WIF         :", auth["name"], "->", sorted(auth["with"]))
assert {"workload_identity_provider", "service_account"} <= set(auth["with"])
assert "credentials_json" not in WORKFLOW.read_text(encoding="utf-8")

promocion = next(p for p in pasos_cr if p.get("name", "").startswith("Promocionar"))
print("Promoción   :", promocion["name"])
assert all(orden in promocion["run"] for orden in ("docker pull", "docker tag", "docker push"))
assert not any("docker build" in p.get("run", "") for p in pasos_cr)

ejecutables = [linea for p in pasos_cr for linea in p.get("run", "").splitlines() if not linea.lstrip().startswith("#")]
assert not any("--allow-unauthenticated" in linea for linea in ejecutables)
manual = (RAIZ / "api" / "deploy" / "gcp" / "desplegar_manual.sh").read_text(encoding="utf-8")
assert "--allow-unauthenticated" in [t for l in manual.splitlines() if not l.lstrip().startswith("#") for t in l.split()]
print("Acceso público: solo en desplegar_manual.sh, hecho a mano por la persona propietaria")
assert wf["jobs"]["desplegar"]["permissions"]["id-token"] == "write"

## Ejercicio 5. Limpiar los recursos (complemento)

El cuaderno original cierra con la limpieza (`limpiar.sh`). La celda del cuaderno original no puede escribir la confirmación que el guion pide, por lo que esta solución la envía por la entrada estándar.

La siguiente celda borrará el servicio, el repositorio de *Artifact Registry*, el fondo de *WIF* y las cuentas de servicio del curso. El guion exige escribir `si`; aquí se pasa con `printf`. Después elimina las variables del entorno `cloudrun` en *GitHub*.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
printf 'si\n' | bash api/deploy/gcp/limpiar.sh
for NOMBRE in GCP_PROJECT_ID GCP_REGION AR_REPOSITORY CLOUD_RUN_SERVICE GCP_WIF_PROVIDER GCP_DEPLOY_SA GCP_RUNTIME_SA; do
  gh variable delete "${NOMBRE}" --env cloudrun
done
gh variable list --env cloudrun

**Salida esperada.** `Limpieza terminada.` y una lista de variables vacía. El *pool* de *WIF* queda en estado «borrado» durante unos días y su identificador no se puede reutilizar en ese plazo; si se va a repetir la práctica, conviene cambiar `WIF_POOL`. Revise la facturación del proyecto después.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2027.</p>